In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# ====================================================
# 1. KHỞI TẠO SPARK KẾT NỐI VỚI CLUSTER
# ====================================================
print("🚀 Đang khởi động Spark kết nối Cluster (Xử lý Customers)...")
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Cluster_Process_Customers") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "3g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã kết nối Spark thành công!")

🚀 Đang khởi động Spark kết nối Cluster (Xử lý Customers)...


26/03/20 16:17:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 16:17:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/20 16:17:25 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ Đã kết nối Spark thành công!


In [2]:
# ====================================================
# 2. KHAI BÁO ĐƯỜNG DẪN HDFS & ĐỌC DỮ LIỆU
# ====================================================
INPUT_FILE = "/data/raw/customers.csv"
TRANS_FILE = "/data/processed/cleaned_transactions.parquet"
OUTPUT_FILE = "/data/processed/customers_processed.parquet"

print(f"1️⃣ Đang đọc Customers từ HDFS: {INPUT_FILE}")
df_customers = spark.read.csv(INPUT_FILE, header=True, inferSchema=True).dropDuplicates(["customer_id"])

print(f"2️⃣ Đang đọc danh sách khách có giao dịch từ: {TRANS_FILE}")
# Chỉ lấy những ID khách hàng đã thực sự mua đồ
df_trans_users = spark.read.parquet(TRANS_FILE).select("customer_id").distinct()

print("✅ Đọc dữ liệu thành công!")

1️⃣ Đang đọc Customers từ HDFS: /data/raw/customers.csv


AnalysisException: Path does not exist: hdfs://namenode:9000/data/raw/customers.csv

In [ ]:
# ---------------------------------------------------------
# BƯỚC 1: LỌC KHÁCH HÀNG TỪ TRANSACTIONS
# ---------------------------------------------------------
print("⚙️ Đang tiến hành lọc khách hàng ảo (chưa từng mua hàng)...")

# Inner Join: Giữ lại khách hàng hợp lệ
df_joined = df_customers.join(df_trans_users, on="customer_id", how="inner")

print("✅ Đã lọc xong khách hàng hợp lệ!")
# Nếu muốn xem còn lại bao nhiêu khách thật, bỏ comment dòng dưới:
# print("Số lượng khách hàng thực tế:", df_joined.count())

In [3]:
# ---------------------------------------------------------
# BƯỚC 2: XỬ LÝ DỮ LIỆU CHI TIẾT (TRẠNG THÁI)
# ---------------------------------------------------------
print("3️⃣ Đang số hóa cột trạng thái thành viên và nhận thông báo...")

# 2.1. FN và Active: FillNA = 0
df_cleaned = df_joined.fillna({"FN": 0, "Active": 0})

# 2.2. Club Member Status -> Số hóa
df_cleaned = df_cleaned.withColumn("club_member_status", F.upper(F.trim(F.col("club_member_status"))))
df_cleaned = df_cleaned.withColumn(
    "club_status_index",
    F.when(F.col("club_member_status") == "ACTIVE", 0)
     .when(F.col("club_member_status") == "PRE-CREATE", 1)
     .when(F.col("club_member_status") == "LEFT CLUB", 2)
     .otherwise(-1) # Unknown
)

# 2.3. Fashion News Frequency -> Số hóa
df_cleaned = df_cleaned.withColumn("fashion_news_frequency", F.upper(F.trim(F.col("fashion_news_frequency"))))
df_cleaned = df_cleaned.withColumn(
    "news_freq_index",
    F.when(F.col("fashion_news_frequency").isin("NONE", "NO"), 0)
     .when(F.col("fashion_news_frequency") == "MONTHLY", 1)
     .when(F.col("fashion_news_frequency") == "REGULARLY", 2)
     .otherwise(0) # Mặc định là None
)

print("✅ Đã chuyển đổi chữ thành số thành công!")

3️⃣ Đang số hóa cột trạng thái thành viên và nhận thông báo...


NameError: name 'df_joined' is not defined

In [4]:
# ---------------------------------------------------------
# BƯỚC 3: XỬ LÝ CỘT TUỔI (AGE)
# ---------------------------------------------------------
print("⚙️ Đang xử lý cột Tuổi (Age)...")
df_cleaned = df_cleaned.withColumn("age", F.col("age").cast(IntegerType()))

# Lọc nhiễu: Tuổi < 15 hoặc > 100 thì cho thành Null
df_cleaned = df_cleaned.withColumn(
    "age", 
    F.when((F.col("age") < 15) | (F.col("age") > 100), None).otherwise(F.col("age"))
)

# Điền tuổi thiếu bằng Median (Trung vị) toàn cục
median_age = df_cleaned.approxQuantile("age", [0.5], 0.01)[0]
print(f"ℹ️ Tuổi trung vị (Median Age) tính được là: {int(median_age)}. Sẽ dùng để điền vào ô trống.")

df_cleaned = df_cleaned.fillna({"age": int(median_age)})
print("✅ Đã xử lý xong tuổi!")

⚙️ Đang xử lý cột Tuổi (Age)...


NameError: name 'df_cleaned' is not defined

In [5]:
# ---------------------------------------------------------
# BƯỚC 4: LỌC CỘT VÀ LƯU TRỮ LÊN HDFS
# ---------------------------------------------------------
final_cols = [
    "customer_id",  # Key kết nối
    "age",          # Để filter độ tuổi (GenZ, Millenial)
    "FN", 
    "Active", 
    "club_status_index",
    "news_freq_index" 
]

df_final = df_cleaned.select(final_cols)

print("👀 5 dòng dữ liệu chuẩn bị lưu:")
df_final.show(5)

# Lưu Parquet lên HDFS
print("4️⃣ Đang lưu file chuẩn lên HDFS...")
df_final.write.mode("overwrite").parquet(OUTPUT_FILE)
print(f"🎉 Hoàn tất! File Customers chuẩn đã lưu tại: {OUTPUT_FILE}")

NameError: name 'df_cleaned' is not defined

In [ ]:
# ====================================================
# ĐÓNG KẾT NỐI SPARK 
# ====================================================
spark.stop()
print("🛑 Đã ngắt kết nối Spark, giải phóng tài nguyên.")